In [0]:
%run ../delta_function

In [0]:
#bibliothèques à importer
import pandas as pd 
from pyspark.sql import functions as F
from pyspark.sql.functions import col, floor
from pyspark.sql import Window
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col, upper, max, when
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, lit, coalesce
from pyspark.sql.window import Window
from pyspark.sql.functions import col, expr
from pyspark.sql.functions import format_string, year, weekofyear
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev' #dev
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'parameters'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

In [0]:
import yaml

yaml_st2 = '/Workspace/Shared/maite/strasbourg2/mal_maite_code/files/inference/src/optimizers/constraints/strasbourg2.yaml'
yaml_pr1 = '/Workspace/Shared/maite/prouvy1/mal_maite_code/files/inference/src/optimizers/constraints/prouvy1.yaml'
yaml_po1 = '/Workspace/Shared/maite/polisy1/mal_maite_code/files/inference/src/optimizers/constraints/polisy1.yaml'
yaml_ro1 = '/Workspace/Shared/maite/rouen1/mal_maite_code/files/inference/src/optimizers/constraints/rouen1.yaml'
yaml_ng1 = '/Workspace/Shared/maite/nogent1/mal_maite_code/files/inference/src/optimizers/constraints/nogent1.yaml'
yaml_ng2 = '/Workspace/Shared/maite/nogent2/mal_maite_code/files/inference/src/optimizers/constraints/nogent2.yaml'
yaml_bu1 = '/Workspace/Shared/maite/buzau1/mal_maite_code/files/inference/src/optimizers/constraints/buzau1.yaml'
yaml_bo1 = '/Workspace/Shared/maite/bolelemi1/mal_maite_code/files/inference/src/optimizers/constraints/bolelemi1.yaml'

with open(yaml_st2, "r") as f:
    yaml_st2 = yaml.safe_load(f)

with open(yaml_pr1, "r") as f:
    yaml_pr1 = yaml.safe_load(f)

with open(yaml_po1, "r") as f:
    yaml_po1 = yaml.safe_load(f)

with open(yaml_ro1, "r") as f:
    yaml_ro1 = yaml.safe_load(f)

with open(yaml_ng1, "r") as f:
    yaml_ng1 = yaml.safe_load(f)

with open(yaml_ng2, "r") as f:
    yaml_ng2 = yaml.safe_load(f)

with open(yaml_bu1, "r") as f:
    yaml_bu1 = yaml.safe_load(f)

with open(yaml_bo1, "r") as f:
    yaml_bo1 = yaml.safe_load(f)


In [0]:
import yaml
import pandas as pd
import os
import re

In [0]:
yaml_files = {
    'STRASBOURG2': yaml_st2,
    'PROUVY1': yaml_pr1,
    'POLISY1': yaml_po1,
    'ROUEN1': yaml_ro1,
    'NOGENT1': yaml_ng1,
    'NOGENT2': yaml_ng2,
    'BUZAU1': yaml_bu1,
    'BOLELEMI1': yaml_bo1
}

In [0]:

def extract_feature_keyword(description):
    if not description:
        return None

    patterns = [
        # 1️Features techniques (les plus précises)
        r'\b(?:germ_|steep_|kiln_)\S+',

        # Process + cellule (ex: Germ C2)
        r'\b(?:Germ|Kiln|Steep)\s+C\d\b',

        # Fallback process seul (ex: Germ)
        r'\b(?:Germ|Kiln|Steep)\b'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, description, re.IGNORECASE)
        if matches:
            # Nettoyage : unique + ordre conservé
            matches = list(dict.fromkeys(matches))
            return ", ".join(matches)

    return None

def extract_goods_specy(condition):
    if not condition:
        return None
    # Regex pour capturer la valeur après goods_specy = '...'
    match = re.search(r"goods_specy\s*=\s*'([^']+)'", condition)
    if match:
        return match.group(1)
    return None

def extract_production_type(condition):
    if not condition:
        return None
    match = re.search(r"production_type\s*=\s*'([^']+)'", condition)
    if match:
        return match.group(1)
    return None

def load_yaml_file(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def flatten_yaml_entries(yaml_dict, source_name=None):
    rows = []

    for section_key, entries in yaml_dict.items():
        if isinstance(entries, list):
            for entry in entries:

                description = entry.get('description', '')
                feature_name = extract_feature_keyword(description)
                condition = str(entry.get('condition', None))

                # Gestion des bornes standard
                borne_min = entry.get('min_delta', entry.get('edge_min', None))
                borne_max = entry.get('max_delta', entry.get('edge_max', None))

                # Gestion des règles définies avec un delta uniquement
                delta = entry.get('delta')
                if delta is not None and borne_min is None and borne_max is None:
                    borne_min = -delta
                    borne_max = delta

                row = {
                    'section': section_key,
                    'features': feature_name,
                    'description': description,
                    'borne_min': borne_min,
                    'borne_max': borne_max,
                    'columns': (
                    next(iter(entry['columns'].keys()), None)
                    if isinstance(entry.get('columns'), dict)
                    else ', '.join(entry['columns'])
                    if isinstance(entry.get('columns'), list) and entry['columns']
                    else None
                ),
                    'condition': condition,
                    'goods_specy': extract_goods_specy(condition),
                    'production_type': extract_production_type(condition),
                    'source': source_name
                }

                rows.append(row)

    return rows


all_rows = []
for site_name, yaml_data in yaml_files.items():  
    site_rows = flatten_yaml_entries(yaml_data, source_name=site_name)
    all_rows.extend(site_rows)


df = pd.DataFrame(all_rows)

df = df.rename(columns={'source': 'prd_line'})

In [0]:
cols_to_clean = ["prd_line", "section", "features", "goods_specy", "production_type"]

for c in cols_to_clean:
    df[c] = df[c].astype(str).fillna("null") #ajout

In [0]:
df = spark.createDataFrame(df)

In [0]:
dim_features_constraint = df.withColumn("borne_min", F.when(F.col("borne_min").isNull(), F.lit(0)).otherwise(F.col("borne_min")))
dim_features_constraint = df.withColumn("borne_max", F.when(F.col("borne_max").isNull(), F.lit(0)).otherwise(F.col("borne_max")))

IMPORT

In [0]:
current_process = "dim_features_constraint"

In [0]:
target_dim_features_constraint = current_catalog +"."+current_schema+"."+current_process
print(target_dim_features_constraint)

In [0]:
all_columns =  dim_features_constraint.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'prd_line'
    ,'section'
    ,'features'
    ,'goods_specy'
    ,'production_type']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    dim_features_constraint, 
    target_dim_features_constraint, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode  # Use "update" for update mode, "full" for delete/insert mode
    )